# pyinfercnv — Phase 3 quickstart (BayesNet + step 20 proxy + mask non-DE + denoise)

End-to-end Phase 3 on the bundled oligodendroglioma downsampled fixture
(184 cells × 6,577 genes). Builds on `tutorial_phase1.ipynb` (preprocess /
smoothing / centering) and `tutorial_phase2.ipynb` (subclustering / HMM /
cnv_regions). The notebook:

1. Re-loads the same fixture and runs `pyinfercnv.infercnv(...)` once with
   **all Phase 3 toggles on** (`HMM=True, BayesMaxPNormal=0.5,
   mask_nonDE_genes=True, denoise=True, reassignCNVs=False`).
2. Inspects the four new `InferCNVResult` fields populated by
   `pipeline_phase3.run_phase3`: `bayes_posterior`, `de_mask`,
   `hmm_proxy_matrix`, `denoised_matrix`.
3. Pre-denoise vs post-denoise heatmap to make denoise visually concrete.
4. **R vs Py side-by-side** for step 20 (state→CN proxy), step 21
   (mask_non_DE), and step 22 (denoise) using the R fixture TSVs under
   `tests/r_out/` — bit-exactness made visible, not just a number.
5. Demonstrates the **fail-loud guards** the orchestrator raises when the
   R-default unsupported branches are requested (`reassignCNVs=True`,
   `noise_logistic=True`).
6. Performance evidence + 0.3 backlog pointer.

**R-parity at a glance** (Phase 3 only):

| step | py module | max_diff vs R | status |
|---|---|---|---|
| step 18 BayesNet Gibbs | `bayesnet/gibbs` | soft-tier (\|ΔP\|<0.10) ≥90% | tier-3.5 (Gibbs MCMC; R uses rjags) |
| step 19 filterHighPNormals | `bayesnet/filter_high_p_normals` | post-filter Jaccard 0.979 | tier-3.5 |
| step 20 state→CN proxy (i6+i3) | `pipeline_phase3._step20_assign_states_to_proxy_expr_vals` | **0.000e+00** | bit-exact |
| step 21 mask_non_DE | `mask_de/wilcoxon` | **1.110e-16** | bit-exact |
| step 22 denoise | `denoise/ref_mean_sd` | bit-exact | bit-exact |

Numbers come from `tests/test_r_parity.py` against R fixture TSVs in
`tests/r_out/`. Re-run `Rscript tests/r_reference.R` to refresh.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import anndata as ad

import pyinfercnv
from pyinfercnv import InferCNVConfig, infercnv
from pyinfercnv.io.genome import load_gene_positions
from pyinfercnv.viz import plot_cnv_heatmap

print("pyinfercnv version:", pyinfercnv.__version__)

## 1. Load the R fixture (identical to phase 1/2 setup)

In [ ]:
FIXTURE_DIR = Path(
    "/media/jason/T7/rerbulid/infercnv/infercnv-master/inst/extdata"
)
COUNTS_PATH = FIXTURE_DIR / "oligodendroglioma_expression_downsampled.counts.matrix.gz"
ANNOT_PATH = FIXTURE_DIR / "oligodendroglioma_annotations_downsampled.txt"

counts_gxc = pd.read_csv(COUNTS_PATH, sep="\t", index_col=0)
annot = pd.read_csv(
    ANNOT_PATH, sep="\t", header=None, names=["cell_id", "annotation"]
).set_index("cell_id")

cells = list(counts_gxc.columns)
X = counts_gxc.T.values.astype(np.float32)
adata = ad.AnnData(
    X=X, obs=annot.reindex(cells),
    var=pd.DataFrame(index=counts_gxc.index),
)
adata.obs.index.name = None
adata.var.index.name = None
adata.layers["counts"] = adata.X.copy()

gene_pos = load_gene_positions("hg38").drop_duplicates(
    subset="gene_symbol", keep="first"
)
var = adata.var.join(
    gene_pos.set_index("gene_symbol")[["chromosome", "start", "end"]],
    how="left",
)
for col in ("chromosome", "start", "end"):
    adata.var[col] = var[col].values

print(adata)
print("\nannotation counts:\n", adata.obs["annotation"].value_counts())

## 2. Run with **all Phase 3 toggles on**

Top-level `infercnv()` invokes `run_phase3()` automatically when `HMM=True`
or any of `BayesMaxPNormal>0`, `mask_nonDE_genes`, `denoise` is set
(`pipeline.py`). The R-faithful step 20 (`inferCNV_ops.R:1463-1499`) fires
whenever HMM states exist, regardless of the other toggles.

**`reassignCNVs=False` is mandatory** for now: the R default is `True`, but
the Python Gibbs port (`bayesnet/gibbs.py`) only implements the
`removeCNV`-only branch. The orchestrator raises `NotImplementedError`
rather than silently dropping the kwarg.

In [ ]:
REFERENCE_CATS = ["Microglia/Macrophage", "Oligodendrocytes (non-malignant)"]

cfg = InferCNVConfig(
    cutoff=1.0,                  # smart-seq2 fixture; use 0.1 for 10x UMI
    HMM=True,
    HMM_type="i6",
    BayesMaxPNormal=0.5,         # R default; turns on step 18 + 19
    reassignCNVs=False,          # see markdown above
    mask_nonDE_genes=True,       # turns on step 21
    denoise=True,                # turns on step 22
    cluster_by_groups=True,
    tumor_subcluster_partition_method="leiden",
    num_threads=1,
    prune_outliers=True,
    random_state=42,
)
cfg.validate()
print(cfg)

In [ ]:
result = infercnv(
    adata,
    config=cfg,
    reference_key="annotation",
    reference_cat=REFERENCE_CATS,
    inplace=False,
)
assert result is not None
print("=== Phase 3 fields populated ===")
for fname in ("bayes_posterior", "de_mask", "hmm_proxy_matrix", "denoised_matrix"):
    val = getattr(result, fname)
    if val is None:
        print(f"  {fname:20s}: None")
    else:
        print(f"  {fname:20s}: shape={val.shape}, dtype={val.dtype}")

## 3. The four new fields

### 3.1 `bayes_posterior` — BayesNet per-region per-state probabilities

Shape `(n_regions, K)` where `K=6` (i6) or `3` (i3). float64. Persisted
from the canonical `cnv_posterior` key returned by
`run_bayesnet_gibbs` (`gibbs.py:269-276`). Each row is one CNV region
(matched to `result.cnv_regions`); each column is a posterior probability
over the K HMM states, summing to ~1 per row.

In [ ]:
bayes_post_df = pd.DataFrame(
    result.bayes_posterior,
    columns=[f"P(state={i})" for i in range(result.bayes_posterior.shape[1])],
)
print(f"bayes_posterior: {result.bayes_posterior.shape[0]} regions \u00d7 "
      f"{result.bayes_posterior.shape[1]} states")
print(f"row sums (sanity, should all be ~1): "
      f"min={bayes_post_df.sum(axis=1).min():.4f} "
      f"max={bayes_post_df.sum(axis=1).max():.4f}")
print(f"P(state=2) = P(neutral) — fraction of regions where neutral wins:")
neutral_idx = 2  # i6 neutral, 0-based
print(f"  argmax==neutral fraction: "
      f"{(result.bayes_posterior.argmax(axis=1) == neutral_idx).mean():.3f}")
bayes_post_df.head(10)

### 3.2 `de_mask` — mask_non_DE gene mask

Shape `(n_cells, n_bins)` bool. `True` = kept (gene is DE between
tumor subcluster and reference); `False` = masked (gene is replaced
with `center_val` in `cnv_matrix_fc` to flatten non-DE noise).

In [ ]:
kept_frac = result.de_mask.mean()
kept_per_cell = result.de_mask.mean(axis=1)
print(f"de_mask: shape={result.de_mask.shape}, dtype={result.de_mask.dtype}")
print(f"  overall kept fraction: {kept_frac:.3f}")
print(f"  per-cell kept fraction: "
      f"min={kept_per_cell.min():.3f} median={np.median(kept_per_cell):.3f} "
      f"max={kept_per_cell.max():.3f}")

### 3.3 `hmm_proxy_matrix` — step 20 state→CN-ratio proxy

Shape `(n_cells, n_bins)` float64. Equivalent to R's
`hmm.infercnv_obj@expr.data` after step 20 (`inferCNV_ops.R:1463-1499`).
i6 lookup is `[0.0, 0.5, 1.0, 1.5, 2.0, 3.0]` (states 0..5, neutral idx 2);
i3 lookup is `[0.5, 1.0, 1.5]`. **Bit-exact vs R** (`max_diff = 0`).

In [ ]:
from pyinfercnv.pipeline_phase3 import _I6_STATE_TO_CN
print(f"i6 state \u2192 CN ratio lookup: {list(_I6_STATE_TO_CN)}")
print(f"hmm_proxy_matrix: shape={result.hmm_proxy_matrix.shape}, "
      f"dtype={result.hmm_proxy_matrix.dtype}")
uniq, counts = np.unique(result.hmm_proxy_matrix, return_counts=True)
print("unique CN values and frequencies:")
for v, c in zip(uniq, counts):
    print(f"  CN={v:.1f}  n={c} ({c/result.hmm_proxy_matrix.size:.1%})")

### 3.4 `denoised_matrix` — reference-mean denoised CNV matrix

Shape `(n_cells, n_bins)` float32. Values inside the reference
mean ± sd band are flattened to the reference mean (R
`clear_noise_via_ref_mean_sd`, `inferCNV_ops.R:2302-2346`). Kept
**separate** from `result.cnv_matrix_fc` so the pre/post comparison is
non-destructive — you can keep both in memory and diff them.

In [ ]:
pre = result.cnv_matrix_fc
post = result.denoised_matrix
delta = np.abs(post - pre)
print(f"pre  (cnv_matrix_fc):    shape={pre.shape}, dtype={pre.dtype}, "
      f"min={pre.min():.3f} max={pre.max():.3f}")
print(f"post (denoised_matrix):  shape={post.shape}, dtype={post.dtype}, "
      f"min={post.min():.3f} max={post.max():.3f}")
print(f"|post - pre|: median={np.median(delta):.4f} "
      f"max={delta.max():.4f}")
print(f"fraction of cells\u00d7bins flattened to reference mean: "
      f"{(delta > 0).mean():.3f}")

## 4. Pre-denoise vs post-denoise heatmap

Visual confirmation of what step 22 did. The post-denoise panel should
show flatter intra-chromosome regions (reference-band noise collapsed to
the centre) while preserving the high-amplitude CNV signal at the cells
and chromosomes where the HMM called non-neutral states.

In [ ]:
fig, axes = plt.subplots(
    2, 1, figsize=(12, max(8, result.cnv_matrix.shape[0] / 18)),
    gridspec_kw={"height_ratios": [1, 1]}, sharex=True,
)

im0 = axes[0].imshow(
    np.log2(np.maximum(pre, 1e-3)),
    aspect="auto", cmap="RdBu_r", vmin=-0.3, vmax=0.3,
    interpolation="nearest",
)
axes[0].set_title("pre-denoise log2(cnv_matrix_fc)")
axes[0].set_ylabel("cells")
fig.colorbar(im0, ax=axes[0], fraction=0.02, pad=0.01)

im1 = axes[1].imshow(
    np.log2(np.maximum(post, 1e-3)),
    aspect="auto", cmap="RdBu_r", vmin=-0.3, vmax=0.3,
    interpolation="nearest",
)
axes[1].set_title("post-denoise log2(denoised_matrix) \u2014 ref \u00b1 sd flattened to mean")
axes[1].set_ylabel("cells")
axes[1].set_xlabel("genomic bins")
fig.colorbar(im1, ax=axes[1], fraction=0.02, pad=0.01)
plt.tight_layout()
plt.show()

## 5. R vs Py side-by-side — bit-exactness made visible

The R reference outputs live under `tests/r_out/` (re-emit with
`Rscript tests/r_reference.R`). Every R TSV here is genes × cells
1-indexed; pyinfercnv stores cells × bins 0-indexed.

**Step 20 (state→CN proxy).** To isolate the lookup contract from
gene-order drift (the in-notebook `result.hmm_proxy_matrix` is built on
hg38-joined genes; `r_reference.R` uses the R fixture's bundled gencode
table — so the gene subsets differ), we apply the Py step-20 lookup
*directly* to R's step-17 HMM state matrix. This is exactly what
`tests/test_r_parity.py::test_step20_state_proxy_parity` verifies.
Expected `max_diff = 0.000e+00`.

In [ ]:
REPO_ROOT = Path("/media/jason/T7/rerbulid/pyinfercnv")
R_OUT = REPO_ROOT / "tests" / "r_out"

# tests/r_out/ is gitignored — the TSVs are regenerated artifacts. If you
# git-cloned and haven't run R yet, this cell will raise; run::
#
#     Rscript tests/r_reference.R
#
# (requires R + the upstream infercnv package). The .executed copy of this
# notebook ships with the rendered figures.
if not (R_OUT / "step17_hmm_i6.tsv").exists():
    raise FileNotFoundError(
        f"R fixture missing: {R_OUT}/step17_hmm_i6.tsv\n"
        "Re-emit with `Rscript tests/r_reference.R` (R + infercnv required), "
        "or view the bundled .executed.ipynb for the rendered output."
    )

# Load R's step17 HMM i6 state matrix (genes × cells, 1-indexed) and apply
# the Py step20 lookup to it. This is exactly what
# tests/test_r_parity.py::test_step20_state_proxy_parity verifies, and
# isolates the lookup contract from any in-notebook gene-order drift —
# the in-notebook `result.hmm_proxy_matrix` may have a different gene
# subset than r_reference.R because the notebook joins gene_pos via
# hg38 instead of the R fixture's bundled gencode table.
from pyinfercnv.pipeline_phase3 import _step20_assign_states_to_proxy_expr_vals


class _MockConfig:
    HMM_type = "i6"


class _MockResult:
    pass


r17 = pd.read_csv(R_OUT / "step17_hmm_i6.tsv", sep="\t", index_col=0)
r17_states_1based = r17.to_numpy(dtype=np.int32)  # (n_genes, n_cells)

# Py side: hmm_states layout is (n_cells, n_bins) and 0-indexed.
py_states_0based = (r17_states_1based.T - 1).astype(np.int8)
mock = _MockResult()
mock.hmm_states = py_states_0based
mock.hmm_states_i3 = None

py20 = _step20_assign_states_to_proxy_expr_vals(mock, _MockConfig())  # cells × genes
py20_genes_x_cells = py20.T

r20 = pd.read_csv(R_OUT / "step20_proxy_i6.tsv", sep="\t", index_col=0)
r20_mat = r20.to_numpy(dtype=np.float64)

assert py20_genes_x_cells.shape == r20_mat.shape, (
    f"shape: py {py20_genes_x_cells.shape} vs r {r20_mat.shape}"
)
diff = np.abs(py20_genes_x_cells - r20_mat).max()
print(f"step20 R vs Py max_diff = {diff:.3e}  (bit-exact contract: < 1e-10)")

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
axes[0].imshow(r20_mat, aspect="auto", cmap="RdBu_r", vmin=0, vmax=2,
               interpolation="nearest")
axes[0].set_title(f"R step 20 proxy (i6) — {r20_mat.shape[0]} genes × {r20_mat.shape[1]} cells")
axes[0].set_xlabel("cells"); axes[0].set_ylabel("genes")
axes[1].imshow(py20_genes_x_cells, aspect="auto", cmap="RdBu_r",
               vmin=0, vmax=2, interpolation="nearest")
axes[1].set_title(f"Py step 20 proxy (i6) — max_diff = {diff:.1e}")
axes[1].set_xlabel("cells")
plt.tight_layout(); plt.show()

**Step 22 (denoise)** — reference-band flattening. R `step22_denoised.tsv`
is also genes × cells. `tests/test_r_parity.py::test_step22_denoise_parity`
establishes the bit-exact contract; here we just look at the R fixture
for completeness.

In [ ]:
if not (R_OUT / "step22_denoised.tsv").exists():
    print(f"R fixture missing: {R_OUT}/step22_denoised.tsv — skipping step 22 panel.")
    print("Re-emit with `Rscript tests/r_reference.R` (R + infercnv required).")
else:
    r22 = pd.read_csv(R_OUT / "step22_denoised.tsv", sep="\t", index_col=0)
    r22_mat = r22.to_numpy(dtype=np.float64)
    print(f"R step22 denoised: shape={r22_mat.shape}, "
          f"min={r22_mat.min():.3f} max={r22_mat.max():.3f}")

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.imshow(np.log2(np.maximum(r22_mat, 1e-3)), aspect="auto",
              cmap="RdBu_r", vmin=-0.3, vmax=0.3, interpolation="nearest")
    ax.set_title("R step 22 denoised log2 (oligo fixture, from tests/r_out/)")
    ax.set_xlabel("cells"); ax.set_ylabel("genes")
    plt.tight_layout(); plt.show()

## 6. Fail-loud guards — R-default unsupported branches

pyinfercnv 0.2.0 is honest about what it does and doesn't implement.
When you ask for an R-default branch that hasn't been ported, the
orchestrator raises **before** any heavy work, with an explicit message
naming the supported alternative. This avoids the silent-divergence
trap (R says it's running branch X, Py is actually running branch Y,
results don't match, nobody notices).

In [ ]:
demos = [
    ("reassignCNVs=True (R default; Py removeCNV-only)",
     dict(HMM=True, BayesMaxPNormal=0.5, reassignCNVs=True)),
    ("denoise=True + noise_logistic=True (R sigmoidal mask)",
     dict(HMM=True, denoise=True, noise_logistic=True)),
    ("BayesMaxPNormal>0 without HMM",
     dict(HMM=False, BayesMaxPNormal=0.5)),
    ("mask_nonDE_genes=True without HMM",
     dict(HMM=False, mask_nonDE_genes=True)),
]
for label, overrides in demos:
    bad_cfg = InferCNVConfig(cutoff=1.0, **overrides)
    try:
        infercnv(adata, config=bad_cfg,
                 reference_key="annotation", reference_cat=REFERENCE_CATS,
                 inplace=False)
    except (NotImplementedError, ValueError) as exc:
        first_line = str(exc).splitlines()[0][:140]
        print(f"\u2713 {label}\n    \u2192 {type(exc).__name__}: {first_line}\n")

## 7. Performance evidence

The Phase 3 BayesNet Gibbs sampler is the heaviest stage in this
notebook (1000 burnin + 1000 samples × 3 chains, all regions). On the
184-cell oligodendroglioma fixture it ports R `rjags` (~8.8 min observed
in `tests/test_r_parity.py::test_step18_bayesnet_parity` print output)
into pure-Python numba JIT (~0.3 s on the same hardware). Mask + denoise
are both `O(n_cells * n_bins)` numpy.

Broader py-vs-R wallclock evidence on five 3CA cancer cohorts (DCIS1,
TNBC1, TNBC3, plus Gao/Kim/Lee/Obradovic/Qian datasets at 0.5–1.4
kilocells per patient): observed 25–192× py-vs-R speedup. See
`scripts/phase2_benchmark/` for the full harness; aggregate output is
in CHANGELOG `## 0.2.0` "Performance". This is **not** a real-world
validation benchmark — both pipelines consume the same upstream 3CA
auto-annotations, so consistency does not imply correctness against
external CNV ground truth.

In [ ]:
if result.profile is not None:
    rows = []
    for name, entry in result.profile.items():
        rows.append({
            "block": name,
            "wallclock_s": round(entry["wallclock_s"], 4),
            "rss_delta_mb": (round(entry["rss_delta_mb"], 1)
                              if entry.get("rss_delta_mb") is not None else None),
        })
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print(f"\ntotal wallclock (Phase 1+2 only \u2014 Phase 3 not yet profiled): "
          f"{df['wallclock_s'].sum():.3f} s")
else:
    print("no profile dict (debug=True off)")

## Next steps / 0.3 backlog

Branches the 0.2.0 orchestrator currently fail-loud-blocks (set the
listed flag to use the supported branch):

* `BayesNet reassignCNVs=True` (R default; Py uses `removeCNV`-only port)
* `denoise noise_logistic=True` (R sigmoidal mask at `inferCNV_heatmap.R:2783`)
* `BayesNet postMcmcMethod='removeCells'` (R variant at `bayesnet/gibbs.py:181-185`)
* `hspike sim_method='simple'` / `'splatter'` (R alt simulation paths)
* `preprocess threshold='auto'` (R threshold-auto branch)
* `denoise apply_median_filtering` (R alt denoise path)
* Phase 3b variational approximation of Gibbs (`bayesnet/vb` — performance
  optimisation; full-quality Gibbs is the primary path)

All of these will be picked up in the 0.3 cycle. Tracking issues land in
`docs/superpowers/findings/` as they are decided. For day-to-day usage,
the supported configurations cover the canonical R-parity workflow.

**Related reading**

* `tutorial_phase1.ipynb` — preprocess / smoothing / centering
* `tutorial_phase2.ipynb` — subclustering / HMM / cnv_regions
* `examples/r_driver_phase2.R` — R reference driver for side-by-side
* `CHANGELOG.md ## 0.2.0` — the full release notes (Phase 1 + 2 + 3 + 0.3 backlog)
* `NAMESPACE_PARITY.md` — R ↔ Py mapping with tier-by-step status